## Célula 0: Imports Globais

Rode esta célula primeiro para carregar todas as bibliotecas necessárias


In [1]:
import json
import time
import random
import shutil
from pathlib import Path

print("✅ Todas as bibliotecas carregadas com sucesso!")

✅ Todas as bibliotecas carregadas com sucesso!


## Célula 1: Configurações Iniciais

Rode esta célula para configurar os caminhos e as classes alvo

In [ ]:
# ==========================================
# 1. CONFIGURAÇÕES DO PROJETO (SSD / PASCAL VOC)
# ==========================================

# Pega o diretório atual onde este notebook está rodando
BASE_DIR = Path.cwd() 
COCO_DIR = BASE_DIR / "coco_2017"
ANNOTATIONS_FILE = COCO_DIR / "annotations" / "instances_train2017.json"
IMAGES_DIR = COCO_DIR / "train2017"

OUTPUT_DIR = BASE_DIR / "SSD_dataset"
OUTPUT_IMAGES = OUTPUT_DIR / "images"
OUTPUT_LABELS = OUTPUT_DIR / "annotations_xml" # Padrão Pascal VOC

IMAGENS_POR_CLASSE = 3000

# Dicionário do COCO original 
COCO_CLASSES = {
    1: 'person',
    2: 'bicycle',
    3: 'car',
    4: 'motorcycle',
    6: 'bus'       
}

print(f"✅ Diretório base configurado para: {BASE_DIR}")
print(f"🎯 Meta: {IMAGENS_POR_CLASSE} imagens por classe alvo.")

✅ Diretório base configurado para: /home/rcferreira/Documents/TCC/COCO-2017
🎯 Meta: 3000 imagens por classe alvo.


## Célula 2: Funções de Suporte

Rode esta célula para registrar as funções na memória do Jupyter

In [ ]:
# ==========================================
# 2. FUNÇÕES DE SUPORTE E CONVERSÃO
# ==========================================

def setup_directories():
    if OUTPUT_DIR.exists():
        print("🧹 Limpando diretório antigo...")
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
    OUTPUT_LABELS.mkdir(parents=True, exist_ok=True)
    print(f"📁 Diretórios criados em: {OUTPUT_DIR}")

def convert_coco_to_voc(bbox, img_width, img_height):
    # Converte [x_min, y_min, w, h] (COCO) para [x_min, y_min, x_max, y_max] (VOC/SSD)
    x_min, y_min, w, h = bbox
    
    # Programação Defensiva: Garante que a caixa não vaze para fora da foto
    x_min = max(0, int(x_min))
    y_min = max(0, int(y_min))
    x_max = min(img_width, int(x_min + w))
    y_max = min(img_height, int(y_min + h))

    return x_min, y_min, x_max, y_max

def create_xml_content(file_name, img_width, img_height, objects_list):
    # Monta a estrutura do arquivo XML exigido pelo SSD
    xml = f"""<annotation>
    <folder>images</folder>
    <filename>{file_name}</filename>
    <size>
        <width>{img_width}</width>
        <height>{img_height}</height>
        <depth>3</depth>
    </size>
"""
    for obj in objects_list:
        xml += f"""    <object>
        <name>{obj['name']}</name>
        <pose>Unspecified</pose>
        <truncated>0</truncated>
        <difficult>0</difficult>
        <bndbox>
            <xmin>{obj['xmin']}</xmin>
            <ymin>{obj['ymin']}</ymin>
            <xmax>{obj['xmax']}</xmax>
            <ymax>{obj['ymax']}</ymax>
        </bndbox>
    </object>
"""
    xml += "</annotation>"
    return xml

print("✅ Funções auxiliares e de conversão XML carregadas!")

✅ Funções auxiliares carregadas!


## Célula 3: O Algoritmo de Mineração (O Cérebro do Script)

Rode esta célula. É aqui que o JSON é lido e a matemática de balanceamento e sobreposição de imagens acontece na RAM. Nenhuma foto é copiada ainda.

In [ ]:
# ==========================================
# 3. LEITURA E BALANCEAMENTO LÓGICO
# ==========================================

start_time = time.time()
print("⏳ Carregando o arquivo JSON do COCO... (Pode levar 1-2 minutos)")

with open(ANNOTATIONS_FILE, 'r') as f:
    coco_data = json.load(f)
print("✅ JSON carregado na memória!")

images_info = {img['id']: img for img in coco_data['images']}

annotations_by_image = {}
for ann in coco_data['annotations']:
    cat_id = ann['category_id']
    if cat_id in COCO_CLASSES and not ann.get('iscrowd', 0):
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)

selected_image_ids = set()
images_per_class = {cat_id: 0 for cat_id in COCO_CLASSES}
objects_per_class = {cat_id: 0 for cat_id in COCO_CLASSES}

print(f"🎯 Iniciando a busca exata de {IMAGENS_POR_CLASSE} imagens por classe...")

for target_cat_id, class_name in COCO_CLASSES.items():
    for img_id, anns in annotations_by_image.items():
        if images_per_class[target_cat_id] >= IMAGENS_POR_CLASSE:
            break 
            
        has_target_class = any(a['category_id'] == target_cat_id for a in anns)
        
        if has_target_class and img_id not in selected_image_ids:
            selected_image_ids.add(img_id)
            
            classes_in_this_image = set()
            for a in anns:
                c_id = a['category_id']
                if c_id in COCO_CLASSES:
                    objects_per_class[c_id] += 1
                    classes_in_this_image.add(c_id)
                    
            for c_id in classes_in_this_image:
                images_per_class[c_id] += 1

print(f"\n🔥 Mineração concluída em {time.time() - start_time:.2f} segundos!")
print(f"Total de imagens únicas selecionadas para extração: {len(selected_image_ids)}")

⏳ Carregando o arquivo JSON do COCO... (Pode levar 1-2 minutos dependendo da sua RAM)
✅ JSON carregado na memória!
🎯 Iniciando a busca exata de 3000 imagens por classe...

🔥 Mineração concluída em 11.61 segundos!
Total de imagens únicas selecionadas para extração: 10332


### Célula 4: A Extração Física (Cópia e Geração dos XMLs) 
Nesta versão, a extração física é feita em um loop separado para evitar sobrecarregar a RAM. Roda esta célula para copiar as imagens e gerar os XMLs.

In [ ]:
# =============================================
# 4. EXTRAÇÃO DE ARQUIVOS E ESCRITA DOS XMLs
# =============================================

print("⚙️  Preparando diretórios de saída...")
setup_directories()

print("🚀 Iniciando a cópia física das imagens e geração dos .xml (Aguarde...)")
imgs_copiadas = 0

for img_id in selected_image_ids:
    img_data = images_info[img_id]
    file_name = img_data['file_name']
    img_width = img_data['width']
    img_height = img_data['height']
    
    src_img_path = IMAGES_DIR / file_name
    dst_img_path = OUTPUT_IMAGES / file_name
    
    if src_img_path.exists():
        shutil.copy2(src_img_path, dst_img_path)
        imgs_copiadas += 1
    else:
        continue 
        
    xml_name = Path(file_name).stem + ".xml"
    xml_path = OUTPUT_LABELS / xml_name
    
    objects_list = []
    for ann in annotations_by_image[img_id]:
        cat_id = ann['category_id']
        if cat_id in COCO_CLASSES:
            class_name = COCO_CLASSES[cat_id]
            x_min, y_min, x_max, y_max = convert_coco_to_voc(ann['bbox'], img_width, img_height)
            
            objects_list.append({
                'name': class_name,
                'xmin': x_min,
                'ymin': y_min,
                'xmax': x_max,
                'ymax': y_max
            })
            
    xml_content = create_xml_content(file_name, img_width, img_height, objects_list)
    
    with open(xml_path, 'w') as xml_file:
        xml_file.write(xml_content)

print(f"✅ Fim da extração física! {imgs_copiadas} imagens copiadas e anotadas com sucesso.")

⚙️ Preparando diretórios de saída...
📁 Diretórios criados em: /home/rcferreira/Documents/TCC/COCO-2017/FasterRCNN_dataset

🧮 Total de imagens para dividir: 10332
   ┣ 🏋️ Treino (80%): 8265
   ┣ 📏 Validação (10%): 1033
   ┗ 🎯 Teste (10%): 1034

🚀 Iniciando a cópia física e geração dos JSONs (Aguarde...)
✅ Fim da extração! 10332 imagens copiadas organizadas nas pastas com seus respectivos JSONs.


### Célula 5: Relatório Final
Rode esta célula. Ela vai imprimir a tabela com relatório final

In [ ]:
# ===================
# 5. RELATÓRIO FINAL
# ===================

print("\n" + "="*95)
print("📊 ANÁLISE DE DISTRIBUIÇÃO DE CLASSES (SSD PASCAL VOC BALANCEADO)")
print("="*95)
print(f"{'Classe ID':<10} {'Nome da Classe':<15} {'Total de Objetos':<20} {'Total de Imagens':<20} {'Média de Objetos/Imagem'}")
print("-" * 95)

total_objetos_geral = 0
for cat_id, name in COCO_CLASSES.items():
    total_obj = objects_per_class[cat_id]
    total_img = images_per_class[cat_id]
    media = total_obj / total_img if total_img > 0 else 0
    total_objetos_geral += total_obj
    
    print(f"{cat_id:<10} {name:<15} {total_obj:<20} {total_img:<20} {media:.2f}")

print("="*95)
print("\n📈 RESUMO GERAL")
print("="*95)
print(f"✓ Total de imagens processadas fisicamente: {imgs_copiadas}")
print(f"✓ Total de objetos detectados (Bounding Boxes): {total_objetos_geral}")

media_geral = total_objetos_geral / imgs_copiadas if imgs_copiadas > 0 else 0
print(f"✓ Média de objetos por imagem no Dataset: {media_geral:.2f}")
print(f"✓ Número de classes ativas: {len(COCO_CLASSES)}")
print("="*95)


📊 ANÁLISE DE DISTRIBUIÇÃO DE CLASSES (FASTER R-CNN BALANCEADO)
Classe ID  Nome da Classe  Total de Objetos     Total de Imagens     Média de Objetos/Imagem
-----------------------------------------------------------------------------------------------
1          person          41776                8086                 5.17
2          bicycle         6615                 3000                 2.21
3          car             17161                4652                 3.69
4          motorcycle      7172                 3023                 2.37
6          bus             4538                 3000                 1.51

📈 RESUMO GERAL
✓ Total de imagens processadas fisicamente: 10332
✓ Total de objetos detectados (Bounding Boxes): 77262
✓ Média de objetos por imagem no Dataset: 7.48
✓ Número de classes ativas: 5

🎉 Tudo pronto! Pode zipar a pasta 'FasterRCNN_dataset'.


### Célula 6: O Fatiamento (Split) e o Label Map do SSD
Rode esta célula para criar as pastas de treino, validação e teste, e para gerar o label map do SSD. O label map é um arquivo de texto que mapeia os IDs das classes para seus nomes, e é necessário para treinar o modelo SSD.

In [ ]:
# ==========================================================
# 6. SPLIT (TRAIN/VAL/TEST) E GERAÇÃO DO LABEL_MAP.PBTXT
# ==========================================================

random.seed(42)

todas_imagens = [f for f in OUTPUT_IMAGES.iterdir() if f.is_file() and f.suffix == '.jpg']
random.shuffle(todas_imagens)

total = len(todas_imagens)
train_idx = int(total * 0.80)
val_idx = int(total * 0.90)

train_imgs = todas_imagens[0:train_idx]       
val_imgs = todas_imagens[train_idx:val_idx]    
test_imgs = todas_imagens[val_idx:total]    

print(f"🧮 Total de imagens para dividir: {total}")
print(f"   ┣ 🏋️ Treino (80%): {len(train_imgs)}")
print(f"   ┣ 📏 Validação (10%): {len(val_imgs)}")
print(f"   ┗ 🎯 Teste (10%): {len(test_imgs)}")

def mover_split(lista_imagens, split_name):
    img_dest = OUTPUT_IMAGES / split_name
    lbl_dest = OUTPUT_LABELS / split_name
    
    img_dest.mkdir(exist_ok=True)
    lbl_dest.mkdir(exist_ok=True)
    
    for img_path in lista_imagens:
        lbl_path = OUTPUT_LABELS / (img_path.stem + ".xml")
        
        shutil.move(str(img_path), str(img_dest / img_path.name))
        
        if lbl_path.exists():
            shutil.move(str(lbl_path), str(lbl_dest / lbl_path.name))

print("\n📂 Reorganizando os arquivos nas subpastas...")
mover_split(train_imgs, "train")
mover_split(val_imgs, "val")
mover_split(test_imgs, "test")

# No SSD não usamos data.yaml, usamos label_map.pbtxt (Padrão TensorFlow Object Detection)
print("📝 Gerando o arquivo 'label_map.pbtxt'...")

pbtxt_content = ""
for idx, class_name in enumerate(COCO_CLASSES.values(), start=1):
    pbtxt_content += "item {\n"
    pbtxt_content += f"  id: {idx}\n"
    pbtxt_content += f"  name: '{class_name}'\n"
    pbtxt_content += "}\n"

with open(OUTPUT_DIR / "label_map.pbtxt", "w") as f:
    f.write(pbtxt_content)

print("✅ Dataset reestruturado com sucesso para o padrão SSD (Pascal VOC)!")

📦 Iniciando a compactação da pasta 'FasterRCNN_dataset'...

✅ SUCESSO! Arquivo 'FasterRCNN_dataset.zip' gerado.
📍 Localização: /home/rcferreira/Documents/TCC/COCO-2017/FasterRCNN_dataset.zip
⏱️  Tempo de execução: 48.65 segundos.


### Célula 7: Compactação do Dataset
Rode esta célula para compactar as pastas de treino, validação e teste em arquivos ZIP. Isso é útil para facilitar o upload do dataset para a nuvem ou para compartilhamento.

In [ ]:
# ============================
# 7. CONFIGURAÇÃO DA COMPACTAÇÃO
# ============================

DATASET_DIR = BASE_DIR / "SSD_dataset"  
ZIP_OUTPUT = BASE_DIR / "SSD_dataset"  

if DATASET_DIR.exists():
    print(f"📦 Iniciando a compactação da pasta '{DATASET_DIR.name}'...")
    start_time = time.time()
    
    shutil.make_archive(ZIP_OUTPUT, 'zip', DATASET_DIR)
    
    print("\n" + "="*75)
    print(f"✅ SUCESSO! Arquivo '{ZIP_OUTPUT.name}.zip' gerado com sucesso.")
    print(f"📍 Localização: {BASE_DIR / (ZIP_OUTPUT.name + '.zip')}")
    print(f"⏱️  Tempo de execução: {time.time() - start_time:.2f} segundos.")
    print("="*75)
else:
    print(f"❌ Erro: O diretório '{DATASET_DIR}' não foi encontrado no caminho atual.")